# Masked Language Model (MLM) — BERT's Pre-training Task #1

Before BERT, language models were trained left-to-right (like GPT). BERT's key innovation
was the **Masked Language Model**: mask out random tokens and train the model to predict them
using **both** left and right context.

This notebook covers:
1. Why we need MLM — the bidirectional problem
2. The masking strategy (80/10/10 rule)
3. Building a simple tokenizer and corpus
4. Implementing the MLM masking procedure
5. Building a small Transformer encoder for MLM
6. Training and watching the model learn to fill in blanks

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import random
import math

%matplotlib inline
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Part 1: Why Masked Language Modeling?

### The problem with left-to-right models

Standard language models (GPT-style) predict the next token given all previous tokens:

```
P(token_t | token_1, token_2, ..., token_{t-1})
```

This is **unidirectional** — each token can only see what came before it.
For tasks like question answering, you need context from **both** directions.

### Why not just use a bidirectional model?

If you let every token attend to every other token (bidirectional), then in a standard
language modeling setup, each word can indirectly "see itself" through the layers.
The model could trivially predict the target word — it would just learn to copy.

### BERT's solution: Mask and Predict

Instead of predicting the next token, BERT:
1. **Randomly masks 15%** of the input tokens
2. Asks the model to **predict the original token** at each masked position
3. Uses **full bidirectional attention** — every token sees every other token

Since the masked tokens are hidden, the model can't cheat. It must use surrounding
context from both sides to figure out what was masked.

## Part 2: The 80/10/10 Masking Strategy

There's a subtle problem: the `[MASK]` token is used during pre-training but **never**
appears during fine-tuning. This creates a mismatch between the two phases.

BERT's solution: when a token is selected for masking (15% of tokens), apply one of
three strategies:

| Strategy | Probability | Example |
|----------|------------|---------|
| Replace with `[MASK]` | 80% | `the cat sat` → `the [MASK] sat` |
| Replace with random token | 10% | `the cat sat` → `the dog sat` |
| Keep unchanged | 10% | `the cat sat` → `the cat sat` |

In all three cases, the model must predict the **original** token.

- The 80% `[MASK]` case is the main training signal
- The 10% random replacement forces the model to not just rely on seeing `[MASK]`
- The 10% unchanged case biases the representation toward the actual observed word

Let's implement this:

## Part 3: A Simple Tokenizer and Corpus

We'll build a tiny word-level tokenizer and a small corpus to train on.
This keeps things transparent — you can see exactly what the model is learning.

In [ ]:
# Simple word-level tokenizer
class SimpleTokenizer:
    def __init__(self):
        self.special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]
        self.word2idx = {}
        self.idx2word = {}
        for i, tok in enumerate(self.special_tokens):
            self.word2idx[tok] = i
            self.idx2word[i] = tok

        self.pad_id = 0
        self.mask_id = 1
        self.cls_id = 2
        self.sep_id = 3
        self.unk_id = 4

    def build_vocab(self, sentences):
        idx = len(self.special_tokens)
        for sent in sentences:
            for word in sent.lower().split():
                if word not in self.word2idx:
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
                    idx += 1

    def encode(self, sentence, max_len=None):
        tokens = [self.cls_id]  # start with [CLS]
        for word in sentence.lower().split():
            tokens.append(self.word2idx.get(word, self.unk_id))
        tokens.append(self.sep_id)  # end with [SEP]
        if max_len:
            tokens = tokens[:max_len]
            tokens += [self.pad_id] * (max_len - len(tokens))
        return tokens

    def decode(self, token_ids):
        return " ".join(self.idx2word.get(idx, "[?]") for idx in token_ids)

    @property
    def vocab_size(self):
        return len(self.word2idx)

# Training corpus — simple sentences with patterns the model can learn
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the mouse",
    "the dog chased the cat",
    "a bird flew over the house",
    "a bird sat on the tree",
    "the fish swam in the pond",
    "the fish jumped out of the water",
    "the boy kicked the ball",
    "the girl kicked the ball",
    "the boy threw the ball to the dog",
    "the girl threw the ball to the cat",
    "the sun rose in the east",
    "the moon rose in the night",
    "the cat slept on the bed",
    "the dog slept on the floor",
    "a man walked to the store",
    "a woman walked to the park",
    "the bird sang a beautiful song",
    "the children played in the garden",
]

tokenizer = SimpleTokenizer()
tokenizer.build_vocab(corpus)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Sample encoding: {tokenizer.encode('the cat sat on the mat')}")
print(f"Decoded back:    {tokenizer.decode(tokenizer.encode('the cat sat on the mat'))}")

## Part 4: Implementing the MLM Masking Procedure

This is the core of BERT's pre-training. For each input sequence:
1. Select 15% of tokens (excluding special tokens) for prediction
2. For each selected token, apply the 80/10/10 rule
3. Return the masked input and the labels (original tokens at masked positions)

In [ ]:
def create_mlm_batch(token_ids_list, tokenizer, mask_prob=0.15):
    """
    Apply BERT's masking strategy to a batch of token sequences.

    Returns:
        masked_inputs: token ids with masking applied
        labels: original token ids at masked positions, -100 elsewhere
                (-100 is ignored by CrossEntropyLoss)
    """
    masked_inputs = []
    labels = []

    for token_ids in token_ids_list:
        masked = list(token_ids)
        label = [-100] * len(token_ids)  # -100 = ignore in loss

        # Find positions eligible for masking (not special tokens, not padding)
        special = {tokenizer.pad_id, tokenizer.cls_id, tokenizer.sep_id}
        candidates = [i for i, t in enumerate(token_ids) if t not in special]

        # Select 15% of candidate positions
        n_mask = max(1, int(len(candidates) * mask_prob))
        mask_positions = random.sample(candidates, min(n_mask, len(candidates)))

        for pos in mask_positions:
            label[pos] = token_ids[pos]  # target is the original token

            rand = random.random()
            if rand < 0.8:
                # 80%: replace with [MASK]
                masked[pos] = tokenizer.mask_id
            elif rand < 0.9:
                # 10%: replace with random token
                masked[pos] = random.randint(
                    len(tokenizer.special_tokens),
                    tokenizer.vocab_size - 1
                )
            # else 10%: keep unchanged

        masked_inputs.append(masked)
        labels.append(label)

    return torch.tensor(masked_inputs), torch.tensor(labels)

# Demo the masking
max_len = 16
sample_ids = tokenizer.encode("the cat sat on the mat", max_len=max_len)
masked_input, label = create_mlm_batch([sample_ids], tokenizer)

print("Original: ", tokenizer.decode(sample_ids))
print("Masked:   ", tokenizer.decode(masked_input[0].tolist()))
print()
print("Token IDs (original):", sample_ids)
print("Token IDs (masked):  ", masked_input[0].tolist())
print("Labels:              ", label[0].tolist())
print()
print("(Labels: -100 means 'ignore', other values are the tokens to predict)")

## Part 5: Building a Small Transformer Encoder for MLM

We'll build a minimal BERT-like model:
- Token embeddings + position embeddings (learned, like BERT)
- A few Transformer encoder layers (bidirectional self-attention)
- An MLM head that projects hidden states back to vocabulary logits

This is intentionally small so it trains fast on CPU.

In [ ]:
class TransformerEncoderLayer(nn.Module):
    """Single Transformer encoder layer with multi-head self-attention + FFN."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # BERT uses GELU, not ReLU
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Self-attention with residual + layer norm
        attn_out, attn_weights = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward with residual + layer norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x, attn_weights


class MiniMLM(nn.Module):
    """
    A minimal BERT-like Masked Language Model.

    Architecture:
    - Token embedding + Learned positional embedding
    - N Transformer encoder layers (bidirectional!)
    - MLM head: LayerNorm -> Linear -> GELU -> Linear(vocab_size)
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2,
                 d_ff=256, max_len=64, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Embeddings (like BERT: token + position, both learned)
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        # Transformer encoder layers
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # MLM prediction head
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size),
        )

    def forward(self, input_ids, key_padding_mask=None):
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device)

        # Combine token + position embeddings
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.emb_norm(self.emb_dropout(x))

        # Pass through Transformer layers
        all_attn_weights = []
        for layer in self.layers:
            x, attn_w = layer(x, key_padding_mask=key_padding_mask)
            all_attn_weights.append(attn_w)

        # Project to vocabulary
        logits = self.mlm_head(x)

        return logits, all_attn_weights

# Create model
d_model = 128
n_heads = 4
n_layers = 2
d_ff = 256
max_len = 16

model = MiniMLM(
    vocab_size=tokenizer.vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_len=max_len,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,} (all trainable)")
print(f"d_model={d_model}, n_heads={n_heads}, n_layers={n_layers}, d_ff={d_ff}")
print(f"Vocab size: {tokenizer.vocab_size}")

## Part 6: Training the MLM

Now we train! For each batch:
1. Encode sentences into token IDs
2. Apply the 80/10/10 masking
3. Feed masked input through the model
4. Compute cross-entropy loss **only at masked positions**
5. Backpropagate and update

The loss should decrease as the model learns word co-occurrence patterns.

In [ ]:
# Prepare dataset
encoded_corpus = [tokenizer.encode(sent, max_len=max_len) for sent in corpus]

# Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

n_epochs = 200
losses = []

model.train()
for epoch in range(n_epochs):
    # Shuffle corpus each epoch
    random.shuffle(encoded_corpus)

    # Create MLM batch from entire corpus
    masked_input, labels = create_mlm_batch(encoded_corpus, tokenizer)
    masked_input = masked_input.to(device)
    labels = labels.to(device)

    # Padding mask: True where padding
    padding_mask = (masked_input == tokenizer.pad_id)

    # Forward pass
    logits, _ = model(masked_input, key_padding_mask=padding_mask)

    # Loss only on masked positions
    loss = criterion(logits.view(-1, tokenizer.vocab_size), labels.view(-1))

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(losses, color='#2980b9', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MLM Loss')
ax.set_title('Training Loss: Learning to Predict Masked Tokens')
ax.axhline(y=np.log(tokenizer.vocab_size), color='red', linestyle='--',
           alpha=0.5, label=f'Random guess: ln({tokenizer.vocab_size}) = {np.log(tokenizer.vocab_size):.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## Part 7: Testing — Can the Model Fill in the Blanks?

Let's mask specific words and see if the model can predict them.
This is the core capability that makes BERT useful for downstream tasks.

In [ ]:
def predict_masked(model, tokenizer, sentence, mask_word, top_k=5):
    """Mask a specific word in a sentence and show top-k predictions."""
    model.eval()

    words = sentence.lower().split()
    if mask_word.lower() not in words:
        print(f"'{mask_word}' not found in sentence")
        return

    # Encode and mask the target word
    token_ids = tokenizer.encode(sentence, max_len=max_len)
    masked_ids = list(token_ids)

    # Find and mask the target word
    target_word_id = tokenizer.word2idx.get(mask_word.lower(), tokenizer.unk_id)
    mask_pos = None
    for i, tid in enumerate(masked_ids):
        if tid == target_word_id:
            mask_pos = i
            masked_ids[i] = tokenizer.mask_id
            break

    if mask_pos is None:
        print(f"Could not find token for '{mask_word}'")
        return

    # Run model
    input_tensor = torch.tensor([masked_ids]).to(device)
    padding_mask = (input_tensor == tokenizer.pad_id)

    with torch.no_grad():
        logits, _ = model(input_tensor, key_padding_mask=padding_mask)

    # Get predictions at masked position
    probs = F.softmax(logits[0, mask_pos], dim=-1)
    top_probs, top_indices = probs.topk(top_k)

    print(f"Input:  {tokenizer.decode(masked_ids)}")
    print(f"Target: {mask_word.lower()}")
    print(f"\nTop {top_k} predictions:")
    for prob, idx in zip(top_probs, top_indices):
        word = tokenizer.idx2word.get(idx.item(), "[?]")
        marker = " ✓" if word == mask_word.lower() else ""
        print(f"  {word:15s} {prob.item():.4f}{marker}")
    print()

# Test on sentences from the training corpus
print("=" * 60)
print("PREDICTIONS ON TRAINING SENTENCES")
print("=" * 60)
print()

predict_masked(model, tokenizer, "the cat sat on the mat", "cat")
predict_masked(model, tokenizer, "the dog chased the cat", "chased")
predict_masked(model, tokenizer, "a bird flew over the house", "bird")
predict_masked(model, tokenizer, "the boy kicked the ball", "kicked")

## Part 8: Visualizing Attention in MLM

One of the powerful things about BERT is that the attention patterns reveal
what the model is "looking at" when predicting a masked token.

Let's visualize the attention weights when the model tries to predict a masked word:

In [ ]:
def visualize_attention(model, tokenizer, sentence, mask_word, layer=0, head=0):
    """Visualize attention weights for a masked prediction."""
    model.eval()

    words = sentence.lower().split()
    token_ids = tokenizer.encode(sentence, max_len=max_len)
    masked_ids = list(token_ids)

    target_word_id = tokenizer.word2idx.get(mask_word.lower(), tokenizer.unk_id)
    for i, tid in enumerate(masked_ids):
        if tid == target_word_id:
            masked_ids[i] = tokenizer.mask_id
            break

    input_tensor = torch.tensor([masked_ids]).to(device)
    padding_mask = (input_tensor == tokenizer.pad_id)

    with torch.no_grad():
        logits, all_attn = model(input_tensor, key_padding_mask=padding_mask)

    # Get attention weights for specified layer and head
    attn = all_attn[layer][0, head].cpu().numpy()  # (seq_len, seq_len)

    # Get token labels (only non-padding)
    token_labels = []
    for tid in masked_ids:
        if tid == tokenizer.pad_id:
            break
        token_labels.append(tokenizer.idx2word.get(tid, "[?]"))

    n = len(token_labels)
    attn = attn[:n, :n]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(token_labels, fontsize=9)
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (attending from)')
    ax.set_title(f'Attention Weights — Layer {layer+1}, Head {head+1}\n'
                 f'Sentence: "{sentence}" (masked: {mask_word})')
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

# Visualize attention for a masked prediction
visualize_attention(model, tokenizer, "the cat sat on the mat", "cat", layer=0, head=0)
visualize_attention(model, tokenizer, "the cat sat on the mat", "cat", layer=1, head=0)

## Summary: Key Takeaways

### What we built
- A **Masked Language Model** from scratch — the core pre-training objective of BERT
- The **80/10/10 masking strategy** that bridges the gap between pre-training and fine-tuning
- A small Transformer encoder that learns to predict masked tokens using bidirectional context

### Why MLM matters

1. **Bidirectional context**: Unlike left-to-right models (GPT), MLM lets every token attend
   to every other token. This is crucial for understanding tasks.

2. **Rich representations**: By learning to predict masked words, the model builds deep
   contextual representations that capture syntax, semantics, and world knowledge.

3. **Transfer learning**: These representations transfer well to downstream tasks.
   Fine-tuning a pre-trained MLM on a small labeled dataset often beats training from scratch.

### What BERT adds on top of this
- **Next Sentence Prediction (NSP)**: A second pre-training task for sentence-pair understanding
- **Segment embeddings**: To distinguish sentence A from sentence B
- **Much larger scale**: BERT-base has 12 layers, 768 hidden dims, 110M parameters
- **Massive pre-training data**: BooksCorpus (800M words) + Wikipedia (2,500M words)

### Connection to the Adapter paper
The adapter paper (Houlsby et al., 2019) takes a pre-trained BERT and asks:
*"Do we really need to fine-tune ALL 110M parameters for each downstream task?"*
Their answer: freeze BERT, add small bottleneck adapter modules, and train only those.
This gives near-identical performance while training only ~3% of the parameters per task.